# Consecutive Empty Seats (Gaps & Islands)

Given a table `b_sql.b_practice.bms` with columns `seat_no` and `is_empty` (`'Y'`/`'N'`), find all empty seats that belong to a group of **3 or more consecutive empty seats**.

Solved two ways:
1. **SQL** — using the classic gaps-and-islands technique (`seat_no - ROW_NUMBER()`).
2. **PySpark** — equivalent logic using the DataFrame API and Window functions.

In [0]:
%sql
-- Create and populate the BMS (empty seats) table
create table b_sql.b_practice.bms (seat_no int ,is_empty varchar(10));
insert into b_sql.b_practice.bms values
(1,'N')
,(2,'Y')
,(3,'N')
,(4,'Y')
,(5,'Y')
,(6,'Y')
,(7,'N')
,(8,'Y')
,(9,'Y')
,(10,'Y')
,(11,'Y')
,(12,'N')
,(13,'Y')
,(14,'Y');


In [0]:
%sql
-- Preview the BMS table
select * from b_sql.b_practice.bms

In [0]:
%sql
-- SQL Method 1: Find consecutive empty seats (3+) using GROUP BY + HAVING
 with cte as (select *, row_number() over (order by seat_no) as rnk,seat_no-row_number() over (order by seat_no) as diff from b_sql.b_practice.bms where is_empty='Y')
 select seat_no from cte where diff in
(select diff from cte group by 1 having count(*)>=3);



In [0]:
%sql
-- SQL Method 2: Find consecutive empty seats (3+) using COUNT(*) OVER (PARTITION BY diff)
WITH cte AS (
    SELECT
        seat_no,
        seat_no - ROW_NUMBER() OVER (ORDER BY seat_no) AS diff
    FROM b_sql.b_practice.bms
    WHERE is_empty = 'Y'
),
cte2 AS (
    SELECT
        seat_no,
        diff,
        COUNT(*) OVER (PARTITION BY diff) AS grp_count
    FROM cte
)
SELECT seat_no
FROM cte2
WHERE grp_count >= 3
ORDER BY seat_no;


In [0]:
# Load the BMS table into a PySpark DataFrame
df=spark.read.table("b_sql.b_practice.bms")
df.display()

In [0]:
# Import PySpark functions, types, and Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# PySpark: Find consecutive empty seats (3+) using gaps-and-islands technique
# diff = seat_no - row_number() groups consecutive seats; filter groups with count >= 3
df_rnk=(df.filter(col("is_empty")=="Y")
        .withColumn("rnk",row_number().over(Window.orderBy(col("seat_no"))))
        .withColumn("diff",col("seat_no")-col("rnk"))
       )
display(df_rnk)
df_filter=df_rnk.groupBy("diff").agg(count("*").alias("cnt")).filter(col("cnt")>=3)
df_final=df_rnk.join(df_filter,df_rnk.diff==df_filter.diff,"inner").select("seat_no")
df_final.display()